In [17]:
# Model download + local smoke test (FLAN‑T5)
# This will download weights the first time you run it (internet required).

import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Auto-select small on CPU, base on GPU (override if you want)
MODEL_CPU = "google/flan-t5-small"
MODEL_GPU = "google/flan-t5-base"
model_id = MODEL_GPU if DEVICE == "cuda" else MODEL_CPU

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

if DEVICE == "cuda":
    model = model.to(DEVICE).half()
else:
    model = model.to(DEVICE)
model.eval()

test_prompt = "what is the solar system"
inputs = tokenizer(test_prompt, return_tensors="pt").to(DEVICE)

with torch.no_grad():
    out_ids = model.generate(**inputs, max_new_tokens=60)
text = tokenizer.decode(out_ids[0], skip_special_tokens=True)

print("Using model:", model_id)
print("Generated text:", text)

Using model: google/flan-t5-small
Generated text: solar system


# Reasoning + feedback module (FLAN‑T5 only)

This notebook contains **only** the FLAN‑T5 side (reasoning / grading / long‑form feedback generation).

**Contract**
- FLAN‑T5 never sees raw images.
- It consumes a **structured text summary** produced from CLIP-based image analysis (see `vision.ipynb`).
- Output is a **single paragraph** of educational feedback (no bullet points).

No custom feedback dataset is assumed; this relies on instruction prompting (no fine-tuning required).

In [ ]:
# Feedback generator (single-paragraph, long-form, supportive, specific)

import re
from typing import Optional

import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

def _pick_flan_model_id(device: str, prefer: Optional[str] = None) -> str:
    if prefer in {"small", "base"}:
        return "google/flan-t5-small" if prefer == "small" else "google/flan-t5-base"
    return "google/flan-t5-base" if device == "cuda" else "google/flan-t5-small"

def load_flan_t5(prefer: Optional[str] = None):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_id = _pick_flan_model_id(device, prefer=prefer)
    tok = AutoTokenizer.from_pretrained(model_id)
    mdl = AutoModelForSeq2SeqLM.from_pretrained(model_id)
    if device == "cuda":
        mdl = mdl.to(device).half()
    else:
        mdl = mdl.to(device)
    mdl.eval()
    return tok, mdl, device, model_id

def build_feedback_prompt(clip_generated_summary: str) -> str:
    return (
        "You are an experienced mathematics teacher.\n\n"
        "Based on the following student answer analysis, write a detailed feedback paragraph.\n"
        "The feedback must:\n"
        "- Be at least 120 words\n"
        "- Be constructive and pedagogical\n"
        "- Clearly state what the student did well\n"
        "- Clearly explain where the student went wrong\n"
        "- Suggest how the student can improve\n\n"
        "Student Answer Analysis:\n"
        f"{clip_generated_summary.strip()}\n\n"
        "Write the feedback in a formal but supportive academic tone.\n"
        "Output must be a single well-structured paragraph with no bullet points."
    )

def postprocess_single_paragraph(text: str) -> str:
    # Remove bullet-like prefixes and collapse whitespace/newlines into one paragraph.
    text = text.strip()
    text = re.sub(r"(?m)^\s*[-*•]+\s+", "", text)
    text = re.sub(r"\s*\n\s*", " ", text)
    text = re.sub(r"\s{2,}", " ", text)
    return text.strip()

@torch.no_grad()
def generate_feedback(
    clip_generated_summary: str,
    prefer_model: Optional[str] = None,
    max_new_tokens: int = 220,
    min_new_tokens: int = 160,
    temperature: float = 0.7,
    top_p: float = 0.9,
    repetition_penalty: float = 1.15,
    no_repeat_ngram_size: int = 4,
    seed: Optional[int] = 0,
    ) -> str:
    tok, mdl, device, model_id = load_flan_t5(prefer=prefer_model)
    prompt = build_feedback_prompt(clip_generated_summary)
    inputs = tok(prompt, return_tensors="pt", truncation=True).to(device)
    if seed is not None:
        torch.manual_seed(int(seed))
        if device == "cuda":
            torch.cuda.manual_seed_all(int(seed))
    out_ids = mdl.generate(
        **inputs,
        do_sample=True,
        temperature=float(temperature),
        top_p=float(top_p),
        max_new_tokens=int(max_new_tokens),
        min_new_tokens=int(min_new_tokens),
        repetition_penalty=float(repetition_penalty),
        no_repeat_ngram_size=int(no_repeat_ngram_size),
    )
    text = tok.decode(out_ids[0], skip_special_tokens=True)
    text = postprocess_single_paragraph(text)
    return f"[{model_id} on {device}] {text}"

In [ ]:
# Minimal demo (uses a fake CLIP-produced summary)

demo_summary = """Image Analysis Summary:
- Subject: Mathematics (Algebra)
- Correctness: Partially correct
- Strengths: Logical step progression and correct setup
- Weaknesses: Error in final simplification; sign mistake when combining like terms
- Handwriting: Legible
"""

feedback = generate_feedback(demo_summary, prefer_model=None)
print(feedback)

## How this connects to CLIP (high level)

1) `vision.ipynb` uses CLIP to analyze the handwritten image and choose top-matching probe labels (image ↔ probe similarity).
2) You turn those top probes into a structured *Student Answer Analysis* summary string.
3) This notebook feeds that summary into FLAN‑T5 to generate a single-paragraph, long-form feedback response.